# Extension: bootstrap confidence intervals

The SciPy statistics lecture covers classical t-tests, Wilcoxon tests, and OLS with formula-based ANOVA contrasts. It does **not** cover **bootstrap resampling** for uncertainty.

**Goal:** Re-estimate key effects from the brain-size study with nonparametric bootstrap confidence intervals, and compare them to the usual parametric intervals.


## Why bootstrap?

- Classical CIs assume normality (or large-sample asymptotics).
- Bootstrap resamples the observed data with replacement, recomputes the statistic many times, and uses the empirical distribution of that statistic for intervals.
- Useful when *n* is modest (here *n* = 40) or distributions are skewed.


In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.formula.api import ols
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

brain = pd.read_csv("brain_size.csv", sep=";", na_values=".")
brain.head()


,Unnamed: 0,Gender,FSIQ,VIQ,PIQ,Weight,Height,MRI_Count
0,1,Female,133,132,124,118.0,64.5,816932
1,2,Male,140,150,124,NaN,72.5,1001121
2,3,Male,139,123,150,143.0,73.3,1038437
3,4,Male,133,129,128,172.0,68.8,965353
4,5,Female,137,132,134,147.0,65.0,951545


## 1. Bootstrap CI for the male − female mean VIQ difference

Compare to the usual two-sample t-interval / t-test.


In [2]:
female = brain.loc[brain["Gender"] == "Female", "VIQ"].to_numpy()
male = brain.loc[brain["Gender"] == "Male", "VIQ"].to_numpy()

obs_diff = male.mean() - female.mean()
print(f"Observed male−female VIQ difference: {obs_diff:.3f}")

# Parametric reference: Welch t-test + approximate CI
tt = stats.ttest_ind(male, female, equal_var=False)
# Manual Welch SE for a CI
se = np.sqrt(male.var(ddof=1) / len(male) + female.var(ddof=1) / len(female))
# Use t critical with Welch-Satterthwaite df from scipy's result when available
df_approx = len(male) + len(female) - 2
t_crit = stats.t.ppf(0.975, df_approx)
param_ci = (obs_diff - t_crit * se, obs_diff + t_crit * se)
print("Welch-style t-test:", tt)
print(f"Approx 95% parametric CI: ({param_ci[0]:.3f}, {param_ci[1]:.3f})")

B = 5000
boot_diffs = np.empty(B)
for b in range(B):
    f_s = rng.choice(female, size=len(female), replace=True)
    m_s = rng.choice(male, size=len(male), replace=True)
    boot_diffs[b] = m_s.mean() - f_s.mean()

boot_ci = np.quantile(boot_diffs, [0.025, 0.975])
print(f"Bootstrap 95% percentile CI: ({boot_ci[0]:.3f}, {boot_ci[1]:.3f})")

plt.hist(boot_diffs, bins=40, color="steelblue", edgecolor="white")
plt.axvline(obs_diff, color="black", linestyle="--", label="observed")
plt.axvline(boot_ci[0], color="crimson", linestyle=":")
plt.axvline(boot_ci[1], color="crimson", linestyle=":", label="95% boot CI")
plt.xlabel("male − female mean VIQ")
plt.ylabel("bootstrap replicates")
plt.legend()
plt.title("Bootstrap distribution of gender VIQ difference")
plt.show()


Observed male−female VIQ difference: 5.800
Welch-style t-test: Ttest_indResult(statistic=0.7726161723275011, pvalue=0.44466074519419097)
Approx 95% parametric CI: (-9.397, 20.997)
Bootstrap 95% percentile CI: (-7.800, 20.151)


<ipython-input-1-231f603a5de2>:36: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 2. Bootstrap CI for an OLS coefficient

Fit `VIQ ~ Gender + MRI_Count` and bootstrap the **Gender[T.Male]** coefficient by resampling rows.


In [3]:
def fit_male_coef(df):
    m = ols("VIQ ~ Gender + MRI_Count", data=df).fit()
    return float(m.params["Gender[T.Male]"])

obs_coef = fit_male_coef(brain)
print(f"Observed Gender[T.Male] coefficient: {obs_coef:.3f}")

# Parametric CI from one OLS fit
m_full = ols("VIQ ~ Gender + MRI_Count", data=brain).fit()
print(m_full.conf_int().loc["Gender[T.Male]"].rename({0: "param_low", 1: "param_high"}))

B = 2000
boot_coefs = np.empty(B)
n = len(brain)
for b in range(B):
    idx = rng.integers(0, n, size=n)
    boot_coefs[b] = fit_male_coef(brain.iloc[idx])

coef_ci = np.quantile(boot_coefs, [0.025, 0.975])
print(f"Bootstrap 95% CI for Gender[T.Male]: ({coef_ci[0]:.3f}, {coef_ci[1]:.3f})")

plt.hist(boot_coefs, bins=40, color="darkseagreen", edgecolor="white")
plt.axvline(obs_coef, color="black", linestyle="--", label="OLS estimate")
plt.axvline(coef_ci[0], color="crimson", linestyle=":")
plt.axvline(coef_ci[1], color="crimson", linestyle=":", label="95% boot CI")
plt.xlabel("Gender[T.Male] coefficient")
plt.ylabel("bootstrap replicates")
plt.legend()
plt.title("Bootstrap CI for adjusted gender effect on VIQ")
plt.show()


Observed Gender[T.Male] coefficient: -7.492
param_low    -26.484438
param_high    11.501000
Name: Gender[T.Male], dtype: float64
Bootstrap 95% CI for Gender[T.Male]: (-22.476, 10.454)


<ipython-input-1-a6b15dc6e460>:30: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 3. Takeaway

Bootstrap percentile intervals for the gender VIQ contrast agree with the classical conclusion: **zero remains inside the 95% interval**, so there is no strong evidence of a male–female VIQ difference in this sample—even after a method that does not lean on normality as hard as the t-test.
